
## SQL and Relational Database Analysis
## DfT Accident Data (2017–2020)

### This notebook demonstrates a relational database approach to analysing UK road accident data. 
### It covers data exploration, cleaning, and SQL queries for Tasks 3(a), 3(b), and 3(c).

In [29]:
import sqlite3
import pandas as pd

# Connect to the database
con = sqlite3.connect("accident_data_v1.0.0_2023.db")
con.execute("PRAGMA foreign_keys = ON;")
print("Connected successfully.")

Connected successfully.


## 1. Database Exploration
### Listing all tables and confirming row counts to understand the structure before querying.

In [30]:
# List all tables and confirm row counts
tables = pd.read_sql("""
    SELECT name FROM sqlite_master 
    WHERE type='table' AND name NOT LIKE 'sqlite_%' 
    ORDER BY name
""", con)
print(tables)

# row counts
for table in ["accident", "casualty", "lsoa", "vehicle"]:
    result = pd.read_sql(f"SELECT COUNT(*) AS total FROM {table}", con)
    print(f"{table}: {result['total'][0]:,} rows")

       name
0  accident
1  casualty
2      lsoa
3   vehicle
accident: 461,352 rows
casualty: 600,332 rows
lsoa: 34,378 rows
vehicle: 849,091 rows


## 2. Data Exploration
### Exploring raw values before cleaning — good data science never assumes the data is clean.

In [31]:
# Explore raw data before cleaning
checks = {
    "casualty — age_of_casualty": "SELECT MIN(age_of_casualty) AS min, MAX(age_of_casualty) AS max FROM casualty",
    "vehicle — age_of_driver":    "SELECT MIN(age_of_driver) AS min, MAX(age_of_driver) AS max FROM vehicle",
    "vehicle — engine_capacity":  "SELECT MIN(engine_capacity_cc) AS min, MAX(engine_capacity_cc) AS max FROM vehicle",
    "vehicle — vehicle_type":     "SELECT MIN(vehicle_type) AS min, MAX(vehicle_type) AS max FROM vehicle",
    "accident — speed_limit":     "SELECT MIN(speed_limit) AS min, MAX(speed_limit) AS max FROM accident",
}
# lsoa — confirm no NULL codes exist
result = pd.read_sql("SELECT COUNT(*) AS total, COUNT(lsoa01cd) AS valid FROM lsoa", con)
print(f"lsoa: total={result['total'][0]:,}, valid codes={result['valid'][0]:,}")

for label, query in checks.items():
    result = pd.read_sql(query, con)
    print(f"{label}: min={result['min'][0]}, max={result['max'][0]}")

lsoa: total=34,378, valid codes=34,378
casualty — age_of_casualty: min=-1, max=102
vehicle — age_of_driver: min=-1, max=102
vehicle — engine_capacity: min=-1, max=99999
vehicle — vehicle_type: min=-1, max=99
accident — speed_limit: min=-1, max=70


## 3. Data Cleaning
### Creating SQL views to filter impossible values without modifying the raw tables.


In [32]:
# Create clean views — preserve raw data, fix impossible values
# CASE nullifies bad column values while keeping every row intact
# WHERE only used where the entire record is structurally invalid
con.executescript("""
DROP VIEW IF EXISTS accident_clean;
DROP VIEW IF EXISTS vehicle_clean;
DROP VIEW IF EXISTS casualty_clean;
DROP VIEW IF EXISTS lsoa_clean;

CREATE VIEW accident_clean AS
SELECT *,
    CASE
        WHEN speed_limit < 1 OR speed_limit > 140 THEN NULL
        ELSE speed_limit
    END AS speed_limit_clean
FROM accident;

CREATE VIEW vehicle_clean AS
SELECT *,
    CASE
        WHEN age_of_driver < 0 OR age_of_driver > 120 THEN NULL
        ELSE age_of_driver
    END AS age_of_driver_clean,
    CASE
        WHEN engine_capacity_cc < 0 OR engine_capacity_cc = 99999 THEN NULL
        ELSE engine_capacity_cc
    END AS engine_capacity_clean
FROM vehicle
WHERE vehicle_type > 0;

CREATE VIEW casualty_clean AS
SELECT *,
    CASE
        WHEN age_of_casualty < 0 OR age_of_casualty > 120 THEN NULL
        ELSE age_of_casualty
    END AS age_of_casualty_clean
FROM casualty;

CREATE VIEW lsoa_clean AS
SELECT * FROM lsoa
WHERE lsoa01cd IS NOT NULL;
""")
con.commit()
print("Clean views created successfully.")

Clean views created successfully.


## 4. Verify Clean Views
### Confirming the cleaning worked by comparing raw vs clean minimum values.

In [33]:
# Verify cleaning worked — compare raw vs clean
print("Raw casualty min age:", pd.read_sql("SELECT MIN(age_of_casualty) FROM casualty", con).iloc[0,0])
print("Clean casualty min age:", pd.read_sql("SELECT MIN(age_of_casualty_clean) FROM casualty_clean", con).iloc[0,0])

print("Raw vehicle min engine:", pd.read_sql("SELECT MIN(engine_capacity_cc) FROM vehicle", con).iloc[0,0])
print("Clean vehicle min engine:", pd.read_sql("SELECT MIN(engine_capacity_clean) FROM vehicle_clean", con).iloc[0,0])

print("Raw vehicle min driver age:", pd.read_sql("SELECT MIN(age_of_driver) FROM vehicle", con).iloc[0,0])
print("Clean vehicle min driver age:", pd.read_sql("SELECT MIN(age_of_driver_clean) FROM vehicle_clean", con).iloc[0,0])

Raw casualty min age: -1
Clean casualty min age: 0
Raw vehicle min engine: -1
Clean vehicle min engine: 1
Raw vehicle min driver age: -1
Clean vehicle min driver age: 1


### 5. Task 3a
### Oldest and Youngest driver/rider in the casualty table

In [38]:
#casualty_class = 1 -- Driver or Rider per STATS19 coding framework
query_a = """
SELECT
    MIN(age_of_casualty_clean) AS Youngest_Driver_Rider,
    MAX(age_of_casualty_clean) AS Oldest_Driver_Rider
FROM casualty_clean
WHERE casualty_class = 1
"""

result_a = pd.read_sql(query_a, con)
result_a

,Youngest_Driver_Rider,Oldest_Driver_Rider
0,2,101


### 6. Task 3b
### Total count of vehicle type 8

In [39]:
# vehicle type 8 = Taxi/Private Hire car per STATS19 coding framework
query_b = """
SELECT COUNT(*) AS Total_Type_8_Vehicles
FROM vehicle
WHERE vehicle_type = 8
"""

result_b = pd.read_sql(query_b, con)
result_b

,Total_Type_8_Vehicles
0,17864


### 7. Task 3c
### JOIN across all tables for all Leeds LSOA regions

In [42]:
# LEFT JOIN on casualty preserves all accident-vehicle records
query_c = """
SELECT
    v.age_of_driver_clean      AS driver_age,
    c.age_of_casualty_clean    AS casualty_age,
    a.speed_limit_clean        AS speed_limit,
    v.engine_capacity_clean    AS engine_capacity_cc,
    l.lsoa01cd                 AS lsoa_code,
    l.lsoa01nm                 AS lsoa_name
FROM accident_clean AS a
JOIN vehicle_clean  AS v ON v.accident_index = a.accident_index
LEFT JOIN casualty_clean AS c ON c.accident_index = a.accident_index
JOIN lsoa_clean     AS l ON a.lsoa_of_accident_location = l.lsoa01cd
WHERE l.lsoa01nm LIKE '%Leeds%'
"""

leeds_df = pd.read_sql(query_c, con).reset_index(drop=True)

# Cast to Int64 - this handles NULL values cleanly without float conversion
for col in ["driver_age", "casualty_age", "speed_limit", "engine_capacity_cc"]:
    leeds_df[col] = pd.to_numeric(leeds_df[col], errors='coerce').astype("Int64")
print(f"Total Leeds records: {len(leeds_df)}")
leeds_df.head(10)

Total Leeds records: 12229


,driver_age,casualty_age,speed_limit,engine_capacity_cc,lsoa_code,lsoa_name
0,35,61,30,1390,E01011525,Leeds 095C
1,32,30,40,998,E01011320,Leeds 091D
2,32,32,40,998,E01011320,Leeds 091D
3,30,30,40,1061,E01011320,Leeds 091D
4,30,32,40,1061,E01011320,Leeds 091D
5,39,34,30,<NA>,E01011720,Leeds 029E
6,34,34,30,1910,E01011720,Leeds 029E
7,39,39,70,1242,E01011297,Leeds 030B
8,39,69,70,1242,E01011297,Leeds 030B
9,78,39,70,1991,E01011297,Leeds 030B
